# ⚡ "SHE" — Blender 3D + Optical DoF 4K Pipeline (Lightning AI / Kaggle / Linux GPU)
Self-healing, Colab-free notebook designed for **Lightning AI Studio** (T4 / L4) or **Kaggle** (2× T4):
1. **Persistent Workspace Checkpointing:** Saves cleaned 4K assets, individual PNG frames, encoded chunks, and pre-upscale masters directly to Studio disk (no Google Drive required).
2. **5-Frame GPU Benchmark:** Measures exact `sec/frame` and estimated GPU hours before you commit.
3. **Chunked Render + 4K Master:** Renders `720p_preview` or `1080p_master`, muxes `She.wav`, and Lanczos-upscales to **4K (`3840×2160`)**.

In [ ]:
# Step 1: Verify GPU & set persistent Studio output directory
!nvidia-smi
!sudo apt-get update -qq && sudo apt-get install -y -qq ffmpeg libgl1 libxi6 libxrender1 libxkbcommon0 libsm6 wget || apt-get update -qq && apt-get install -y -qq ffmpeg libgl1 libxi6 libxrender1 libxkbcommon0 libsm6 wget
!pip install -q opencv-python pillow numpy

import os, shutil, subprocess

# Lightning AI persistent studio path (falls back to ./SHE_Film_4K_Blender on Kaggle/other)
BASE_STORAGE = '/teamspace/studios/this_studio' if os.path.exists('/teamspace/studios/this_studio') else os.getcwd()
WORK_DIR = os.path.join(BASE_STORAGE, 'SHE_Film_4K_Blender')
CHUNKS_DIR = os.path.join(WORK_DIR, 'chunks')
os.makedirs(CHUNKS_DIR, exist_ok=True)
print('✓ Persistent output dir:', WORK_DIR)

In [ ]:
# Step 2: Install Blender 4.3.2 (Linux x64) & clone SHE repository
BLENDER_VER = '4.3.2'
BLENDER_ROOT = os.path.join(BASE_STORAGE, f'blender-{BLENDER_VER}-linux-x64')
BLENDER_BIN = os.path.join(BLENDER_ROOT, 'blender')
if not os.path.exists(BLENDER_BIN):
    tar_path = os.path.join(BASE_STORAGE, 'blender.tar.xz')
    url = f'https://download.blender.org/release/Blender4.3/blender-{BLENDER_VER}-linux-x64.tar.xz'
    subprocess.run(['wget', '-q', url, '-O', tar_path], check=True)
    subprocess.run(['tar', '-xf', tar_path, '-C', BASE_STORAGE], check=True)
    os.remove(tar_path)

REPO = os.path.join(BASE_STORAGE, 'aianimationautomate')
if not os.path.exists(os.path.join(REPO, 'She.wav')):
    subprocess.run(['rm', '-rf', REPO], check=False)
    subprocess.run([
        'git', '-c', 'http.version=HTTP/1.1', '-c', 'http.postBuffer=524288000',
        'clone', '--depth', '1', '--branch', 'main', '--single-branch',
        'https://github.com/sayan-bhattacharya/aianimationautomate.git', REPO
    ], check=True)

subprocess.run([BLENDER_BIN, '--version'], check=True)
print('✓ Blender & repo ready')

In [ ]:
# Step 3: Use pre-cleaned 4K assets from repo (or rebuild if missing)
CLEAN_DIR = os.path.join(REPO, 'assets', 'blender_clean')
if not (os.path.exists(CLEAN_DIR) and len(os.listdir(CLEAN_DIR)) >= 9):
    subprocess.run(['python', os.path.join(REPO, 'clean_blender_assets.py')], check=True)
print('✓ Cleaned 4K plates ready:', sorted(os.listdir(CLEAN_DIR)))

In [ ]:
# Step 4: Write Blender Scene Script (Cycles GPU + Optical DoF + Per-Frame Resume)
BLENDER_SCRIPT = os.path.join(WORK_DIR, 'render_she_blender.py')
script_src = '''
import math, os, sys, time, bpy, bmesh
from mathutils import Vector

argv = sys.argv[sys.argv.index("--") + 1:]
OUT_DIR = argv[0]
START_F = int(argv[1])
END_F = int(argv[2])
RES_W = int(argv[3])
RES_H = int(argv[4])
SAMPLES = int(argv[5])
CLEAN_DIR = argv[6]
FPS = 24.0
os.makedirs(OUT_DIR, exist_ok=True)

bpy.ops.wm.read_factory_settings(use_empty=True)
sc = bpy.context.scene
sc.render.engine = "CYCLES"
prefs = bpy.context.preferences.addons["cycles"].preferences
for dev_type in ("OPTIX", "CUDA", "NONE"):
    try:
        prefs.compute_device_type = dev_type
        prefs.get_devices()
        if any(d.type != "CPU" for d in prefs.devices):
            for d in prefs.devices: d.use = True
            sc.cycles.device = "GPU"
            print("Using Cycles GPU:", dev_type)
            break
    except Exception:
        continue

sc.cycles.samples = SAMPLES
sc.cycles.use_denoising = True
sc.render.resolution_x = RES_W
sc.render.resolution_y = RES_H
sc.render.resolution_percentage = 100
sc.render.image_settings.file_format = "PNG"

def dune_height(x, y):
    r1 = 0.75 * math.sin(x * 0.28 + 0.4) * math.exp(-((y - 2.0) ** 2) / 14.0)
    r2 = 1.15 * math.cos(x * 0.19 - 0.8) * math.exp(-((y - 7.5) ** 2) / 18.0)
    return max(0.0, (y + 1.0) * 0.12) + r1 + r2 - 0.35

bm = bmesh.new()
nx, ny = 120, 60
verts = [[bm.verts.new((-22 + 44*i/(nx-1), -1 + 19*j/(ny-1), dune_height(-22 + 44*i/(nx-1), -1 + 19*j/(ny-1)))) for i in range(nx)] for j in range(ny)]
for j in range(ny-1):
    for i in range(nx-1):
        bm.faces.new((verts[j][i], verts[j][i+1], verts[j+1][i+1], verts[j+1][i]))
mesh = bpy.data.meshes.new("Dunes")
bm.to_mesh(mesh); bm.free()
for p in mesh.polygons: p.use_smooth = True
dunes = bpy.data.objects.new("Dunes", mesh)
bpy.context.collection.objects.link(dunes)
smat = bpy.data.materials.new("Sand")
smat.use_nodes = True
smat.node_tree.nodes["Principled BSDF"].inputs["Base Color"].default_value = (0.38, 0.31, 0.28, 1.0)
smat.node_tree.nodes["Principled BSDF"].inputs["Roughness"].default_value = 0.88
dunes.data.materials.append(smat)

bpy.ops.mesh.primitive_plane_add(size=80.0, location=(0.0, -18.0, -0.32))
ocean = bpy.context.active_object
omat = bpy.data.materials.new("Ocean")
omat.use_nodes = True
omat.node_tree.nodes["Principled BSDF"].inputs["Base Color"].default_value = (0.012, 0.085, 0.175, 1.0)
omat.node_tree.nodes["Principled BSDF"].inputs["Roughness"].default_value = 0.16
ocean.data.materials.append(omat)

def add_card(name, fname, x, y, h_w, sink=0.20):
    img = bpy.data.images.load(os.path.join(CLEAN_DIR, fname), check_existing=True)
    asp = img.size[0] / max(1, img.size[1])
    bpy.ops.mesh.primitive_plane_add(size=1.0, location=(x, y, dune_height(x, y) + h_w*0.5 - sink))
    c = bpy.context.active_object; c.name = name
    c.scale = (h_w * asp, h_w, 1.0)
    c.rotation_euler = (math.radians(90), 0, 0)
    bpy.ops.object.transform_apply(rotation=True, scale=True)
    m = bpy.data.materials.new(name)
    m.use_nodes = True
    nt = m.node_tree; nt.nodes.clear()
    out = nt.nodes.new("ShaderNodeOutputMaterial")
    bsdf = nt.nodes.new("ShaderNodeBsdfPrincipled")
    tex = nt.nodes.new("ShaderNodeTexImage"); tex.image = img; tex.extension = "CLIP"
    nt.links.new(tex.outputs["Color"], bsdf.inputs["Base Color"])
    nt.links.new(tex.outputs["Alpha"], bsdf.inputs["Alpha"])
    nt.links.new(bsdf.outputs["BSDF"], out.inputs["Surface"])
    c.data.materials.append(m)
    return c

mush = [add_card(f"M_{i}", "hero_pagoda_mushroom_master.png", x, y, h, 0.22) for i, (x, y, h) in enumerate([(-5.2,4.5,2.7),(-2.6,8.2,2.4),(0.2,5.8,2.5),(2.7,2.2,3.2),(5.4,7.5,2.3)])]
firs = [add_card(f"F_{i}", fn, x, y, h, 0.30) for i, (fn, x, y, h) in enumerate([("pine_fuller_warm.png",-5.6,3.0,2.2),("pine_fuller_deep.png",-2.7,4.2,5.4),("pine_fuller_cool.png",0.5,7.2,3.2),("pine_fuller_warm.png",3.4,6.0,3.4)])]
sauro = add_card("Sauro", "pokemon_sauropod.png", 3.1, 3.0, 2.5, 0.12)
hadro = add_card("Hadro", "pokemon_hadrosaur.png", 5.6, 3.4, 1.7, 0.12)
anky = add_card("Anky", "pokemon_ankylosaur.png", 7.6, 4.0, 1.2, 0.10)

world = bpy.data.worlds.new("W"); sc.world = world; world.use_nodes = True
world.node_tree.nodes["Background"].inputs["Color"].default_value = (0.45, 0.38, 0.34, 1.0)
ldata = bpy.data.lights.new("Sun", type="SUN"); ldata.energy = 4.2
sun_l = bpy.data.objects.new("Sun", ldata); bpy.context.collection.objects.link(sun_l)
cam_d = bpy.data.cameras.new("Cam"); cam_d.lens = 42.0; cam_d.dof.use_dof = True; cam_d.dof.focus_distance = 13.5; cam_d.dof.aperture_fstop = 2.0
cam = bpy.data.objects.new("Cam", cam_d); bpy.context.collection.objects.link(cam)
cam.location = (0.0, -10.5, 1.45)
cam.rotation_euler = (Vector((0, 4, 1.25)) - cam.location).to_track_quat("-Z", "Y").to_euler()
sc.camera = cam

t0 = time.time()
skipped = 0
for fi in range(START_F, END_F):
    out_png = os.path.join(OUT_DIR, f"frame_{fi:05d}.png")
    if os.path.exists(out_png) and os.path.getsize(out_png) > 10_000:
        skipped += 1
        continue
    t = fi / FPS
    for o in mush: o.hide_render = not (25.0 <= t < 176.0 or t >= 400.0)
    for o in firs: o.hide_render = not (176.0 <= t < 405.0)
    for o in (sauro, hadro, anky): o.hide_render = not (232.0 <= t < 405.0)
    if not sauro.hide_render:
        p = (t - 232.0) / 173.0
        sx = 8.0 - p * 14.0
        sauro.location.x = sx; hadro.location.x = sx + 2.5; anky.location.x = sx + 4.4
    sun_x = -8.0 + 16.0 * ((t % 120.0) / 120.0)
    sun_l.rotation_euler = (Vector((0, 4, 0)) - Vector((sun_x, 24.0, 5.5))).to_track_quat("-Z", "Y").to_euler()
    sc.render.filepath = out_png
    bpy.ops.render.render(write_still=True)
    if (fi - START_F) % 24 == 0:
        print(f"Rendered frame {fi} (skipped={skipped}, {time.time()-t0:.1f}s elapsed)", flush=True)
print(f"DONE_RANGE {START_F}-{END_F} (skipped={skipped}) in {time.time()-t0:.2f}s", flush=True)
'''
with open(BLENDER_SCRIPT, 'w') as f:
    f.write(script_src.strip())
print('✓ Wrote', BLENDER_SCRIPT)

In [ ]:
# Step 5: 5-Frame GPU Benchmark (720p & 1080p) -> Check exact sec/frame & credit usage
import time
for label, w, h, samp in [('720p (8 samples)', 1280, 720, 8), ('1080p (12 samples)', 1920, 1080, 12)]:
    t0 = time.time()
    subprocess.run([BLENDER_BIN, '-b', '--python', BLENDER_SCRIPT, '--', os.path.join(WORK_DIR, f'bench_{w}'), '6280', '6285', str(w), str(h), str(samp), CLEAN_DIR], check=True)
    dt = (time.time() - t0) / 5.0
    est_hrs = (10680 * dt) / 3600.0
    print(f'⚡ {label}: {dt:.2f} s/frame -> Est. full 10,680 frames = {est_hrs:.2f} hours')

In [ ]:
# Step 6: Self-Healing Chunked Render -> Safe-Point Master -> 4K Output
MODE = '720p_preview'  # Change to '1080p_master' after reviewing 720p
RES_W, RES_H, SAMPLES = (1280, 720, 8) if MODE == '720p_preview' else (1920, 1080, 12)
TOTAL_FRAMES = 10680
CHUNK_SIZE = 1068
n_chunks = (TOTAL_FRAMES + CHUNK_SIZE - 1) // CHUNK_SIZE
chunk_mp4s = []

for ci in range(n_chunks):
    f0, f1 = ci * CHUNK_SIZE, min(TOTAL_FRAMES, (ci + 1) * CHUNK_SIZE)
    c_mp4 = os.path.join(CHUNKS_DIR, f'blender_chunk_{ci+1:02d}_{MODE}.mp4')
    chunk_mp4s.append(c_mp4)
    if os.path.exists(c_mp4) and os.path.getsize(c_mp4) > 100_000:
        print(f'Skipping completed chunk {ci+1:02d}')
        continue
    tmp_frames = os.path.join(WORK_DIR, f'frames_{MODE}', f'chunk_{ci+1:02d}')
    subprocess.run([BLENDER_BIN, '-b', '--python', BLENDER_SCRIPT, '--', tmp_frames, str(f0), str(f1), str(RES_W), str(RES_H), str(SAMPLES), CLEAN_DIR], check=True)
    subprocess.run(['ffmpeg', '-y', '-framerate', '24', '-start_number', str(f0), '-i', f'{tmp_frames}/frame_%05d.png', '-c:v', 'libx264', '-crf', '16', '-pix_fmt', 'yuv420p', c_mp4], check=True)
    shutil.rmtree(tmp_frames, ignore_errors=True)
    print(f'✓ Saved chunk {ci+1:02d}/{n_chunks}:', c_mp4)

concat_txt = os.path.join(WORK_DIR, 'blender_concat.txt')
with open(concat_txt, 'w') as f:
    for cp in chunk_mp4s: f.write(f"file '{cp}'\n")

SAFE_MASTER = os.path.join(WORK_DIR, f'SHE_Blender_DoF_{MODE}_master.mp4')
if not (os.path.exists(SAFE_MASTER) and os.path.getsize(SAFE_MASTER) > 1_000_000):
    subprocess.run([
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', concat_txt,
        '-i', os.path.join(REPO, 'She.wav'),
        '-c:v', 'copy', '-c:a', 'aac', '-b:a', '256k', '-shortest', SAFE_MASTER
    ], check=True)
    print('✓ Safe-point master saved:', SAFE_MASTER)

MASTER_4K = os.path.join(WORK_DIR, f'SHE_Blender_DoF_4K_{MODE}.mp4')
subprocess.run([
    'ffmpeg', '-y', '-i', SAFE_MASTER,
    '-vf', 'scale=3840:2160:flags=lanczos,unsharp=5:5:0.45:5:5:0.0',
    '-c:v', 'libx264', '-preset', 'medium', '-crf', '16', '-pix_fmt', 'yuv420p',
    '-c:a', 'copy', MASTER_4K
], check=True)
print('✓ Final 4K Blender Master ready:', MASTER_4K)